In [4]:
import pandas as pd
import numpy as np
import psycopg
import hashlib

from decimal import Decimal
from pathlib import Path

from getpass import getpass
import psycopg

pg_password = getpass("PostgreSQL password: ")

conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="sales_etl",
    user="postgres",
    password=pg_password
)


PostgreSQL password:  ········


In [5]:
gold_df = pd.read_sql_query(
    """
    SELECT
        customer_id,
        COUNT(*) AS total_orders,
        SUM(amount) AS total_sales
    FROM silver.sales
    GROUP BY customer_id
    """,
    conn
)

print("Gold customer records:", len(gold_df))
display(gold_df.head())

Gold customer records: 10000


C:\Users\Ajay Patil\AppData\Local\Temp\ipykernel_4712\282520353.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  gold_df = pd.read_sql_query(


,customer_id,total_orders,total_sales
0,C07307,7,200859.60
1,C09100,13,282773.88
2,C08197,8,226851.35
3,C01208,11,304186.27
4,C08251,13,327042.69


In [6]:
gold_upsert_sql = """
INSERT INTO gold.sales_summary (
    customer_id,
    total_orders,
    total_sales,
    refreshed_at
)
VALUES (%s, %s, %s, CURRENT_TIMESTAMP)

ON CONFLICT (customer_id)
DO UPDATE SET
    total_orders = EXCLUDED.total_orders,
    total_sales = EXCLUDED.total_sales,
    refreshed_at = CURRENT_TIMESTAMP;
"""

gold_records = [
    (
        str(row.customer_id),
        int(row.total_orders),
        float(row.total_sales)
    )
    for row in gold_df.itertuples(index=False)
]

try:
    with conn.cursor() as cur:
        cur.executemany(gold_upsert_sql, gold_records)
    conn.commit()
    print("Gold upsert completed:", len(gold_records))

except Exception:
    conn.rollback()
    raise

Gold upsert completed: 10000
